In [21]:
include("C:\\Users\\kwiat\\projekty\\julia\\my_AD_library\\src\\my_AD_library.jl")
using Zygote
using Flux
using LinearAlgebra: I

In [25]:
# sin(x^2), x = 5
x = Node(5., true)
f = sin(x * x)
order = topo_sort(f)
backward(f,order)

g(x) = sin(x*x)

println("--- test 1 ---")
println("my_library: value=", f.value, ", grad=", x.grad)
println("    zygote: value=", g(5.), ", grad=", g'(5.))

--- test 1 ---
my_library: value=-0.13235175009777303, grad=9.912028118634735
    zygote: value=-0.13235175009777303, grad=9.912028118634735


In [26]:
# cos(x-2) + sin(x^x) - tan(2x), x = 6.2
x = Node(6.2, true)
c = Node(2., false)
f = cos(x - c) + sin(x*x) - tan(c * x)
order = topo_sort(f)
backward(f,order)

g(x) = cos(x-2.) + sin(x*x) - tan(2x)

println("--- test 2 ---")
println("my_library: value=", f.value, ", grad=", x.grad)
println("    zygote: value=", g(6.2), ", grad=", g'(6.2))

--- test 2 ---
my_library: value=0.35260550296605264, grad=7.964760123533824
    zygote: value=0.35260550296605264, grad=7.964760123533824


In [28]:
# tanh(-x), x = 4.1
x = Node(4.1, true)
f = tanh(-x)
order = topo_sort(f)
backward(f,order)

g(x) = tanh(-x)

println("--- test 3 ---")
println("my_library: value=", f.value, ", grad=", x.grad)
println("    zygote: value=", g(4.1), ", grad=", g'(4.1))

--- test 3 ---
my_library: value=-0.9994508436877974, grad=-0.001098011051750114
    zygote: value=-0.9994508436877974, grad=-0.001098011051750114


In [ ]:
# ∑(Relu(Ax+b))
A = Float32[1 2 3; 9 8 7; 0 0 1]
b = Float32[15, 0, 4]
x = Float32[-1, -2, -3]

weights = Node(A)
fun = sum(ReLU(weights*Node(x)+Node(b)))
order = topo_sort(fun)
backward(fun,order)

g(W) = sum(NNlib.relu(W*x+b))
println("--- ReLU test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
println("    zygote: value=", g(A), ", grad=", g'(A))

--- ReLU test ---
my_library: value=2.0, grad=Float32[-1.0 -2.0 -3.0; 0.0 0.0 0.0; -1.0 -2.0 -3.0]
    zygote: value=2.0, grad=Float32[-1.0 -2.0 -3.0; -0.0 -0.0 -0.0; -1.0 -2.0 -3.0]


In [ ]:
# ∑(sigm(Ax+b))
A = Float32[1 2 3; 9 8 7; 0 0 1]
b = Float32[15, 0, 4]
x = Float32[-1, -2, -3]

weights = Node(A)
fun = sum(sigm(weights*Node(x)+Node(b)))
order = topo_sort(fun)
backward(fun,order)

g(W) = sum(NNlib.sigmoid(W*x+b))
println("--- Sigmoid test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
println("    zygote: value=", g(A), ", grad=", g'(A))

--- Sigmoid test ---
my_library: value=1.4621172, grad=Float32[-0.19661193 -0.39322385 -0.58983576; -1.05306175f-20 -2.1061235f-20 -3.1591853f-20; -0.19661193 -0.39322385 -0.58983576]
    zygote: value=1.4621172, grad=Float32[-0.19661193 -0.39322385 -0.58983576; -1.05306175f-20 -2.1061235f-20 -3.1591853f-20; -0.19661193 -0.39322385 -0.58983576]


In [ ]:
# ∑(tanh(Ax+b))
A = Float32[1 2 3; 9 8 7; 0 0 1]
b = Float32[15, 0, 4]
x = Float32[-1, -2, -3]

weights = Node(A)
fun = sum(tanh(weights*Node(x)+Node(b)))
order = topo_sort(fun)
backward(fun,order)

g(W) = sum(tanh.(W*x+b))
println("--- Tanh test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
println("    zygote: value=", g(A), ", grad=", g'(A))

--- Tanh test ---
my_library: value=0.52318835, grad=Float32[-0.41997433 -0.83994865 -1.259923; 0.0 0.0 0.0; -0.41997433 -0.83994865 -1.259923]
    zygote: value=0.52318835, grad=Float32[-0.41997433 -0.83994865 -1.259923; -0.0 -0.0 -0.0; -0.41997433 -0.83994865 -1.259923]


In [64]:
# BinaryCrossEntropy(sigm(Ax+b), 0)

x = Float64[-1, 2, -3]
A = Float64[4, 5, 1]'
b = 8.

weights = Node(A)
fun = myBinaryCrossEntropy(Node(0.),sigm(weights*Node(x)+ Node(b)))
order = topo_sort(fun)
backward(fun,order)


g(W) = Flux.binarycrossentropy(NNlib.sigmoid(W*x+b),0.)
println("--- BinaryCrossEntropy test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
println("    zygote: value=", g(A), ", grad=", g'(A))

--- BinaryCrossEntropy test ---
my_library: value=11.000016701569812, grad=[-0.999983298578152 1.999966597156304 -2.9999498957344564]
    zygote: value=11.000016701556518, grad=[-0.9999832985648572 1.9999665971297145 -2.999949895694572]


In [36]:
# BinaryCrossEntropy(sigm(Ax+b), 0) for batch training

x = Float64[-1, 2, -3]
A = Float64[4 5 1; 6 -4 1; 9 0 -2]
b = Float64[1, 2, 3]

y = Float64[1, 0, 0]

weights = Node(A)
fun = myBinaryCrossEntropy(Node(y),sigm(weights*Node(x)+ Node(b)))
order = topo_sort(fun)
backward(fun,order)


g(W) = Flux.binarycrossentropy.(NNlib.sigmoid(W*x+b),y)
println("--- BinaryCrossEntropy batch test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
id = Float64[1 0 0; 1 0 0; 1 0 0; 0 1 0; 0 1 0; 0 1 0; 0 0 1; 0 0 1; 0 0 1;]
println("    zygote: value=", g(A), ", grad=", jacobian(g,A)[1]*id)

--- BinaryCrossEntropy batch test ---
my_library: value=[0.01814992791780973, 3.059022736706331e-7, 0.6931471805599453], grad=[0.01798620996209155 -0.0359724199241831 0.053958629886274645; -3.0590222692562467e-7 6.118044538512493e-7 -9.177066807768741e-7; -0.5 1.0 -1.5]
    zygote: value=[0.018149927917809505, 3.059022734485884e-7, 0.6931471805599448], grad=[0.017986209962091545 -0.03597241992418309 0.05395862988627463; -3.059022269256246e-7 6.118044538512492e-7 -9.177066807768738e-7; -0.4999999999999998 0.9999999999999996 -1.4999999999999993]


In [76]:
# MSE(sigm(Ax+b), 0)

x = Float64[-1, 2, -3]
A = Float64[4, 5, 1]'
b = 11.

weights = Node(A)
fun = myMSE(Node(11.),weights*Node(x) + Node(b))
order = topo_sort(fun)
backward(fun,order)


g(W) = Flux.mse((W*x+b),11.)
println("--- MSE test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
println("    zygote: value=", g(A), ", grad=", g'(A))

--- MSE test ---
my_library: value=9.0, grad=[-6.0 12.0 -18.0]
    zygote: value=9.0, grad=[-6.0 12.0 -18.0]


In [37]:
# MSE(sigm(Ax+b), 0)

x = Float64[-1, 2, -3]
A = Float64[4 5 1; 6 -4 1; 9 0 -2]
b = Float64[1, 2, 3]

y = Float64[2, -12, 4]

weights = Node(A)
fun = myMSE(Node(y),weights*Node(x) + Node(b))
order = topo_sort(fun)
backward(fun,order)


g(W) = (y .- (W*x+b)).^2

println("--- MSE batch test ---")
println("my_library: value=", fun.value, ", grad=", weights.grad)
id = Float64[1 0 0; 1 0 0; 1 0 0; 0 1 0; 0 1 0; 0 1 0; 0 0 1; 0 0 1; 0 0 1;]
println("    zygote: value=", g(A), ", grad=", jacobian(g,A)[1]*id)

--- MSE batch test ---
my_library: value=[4.0, 9.0, 16.0], grad=[-4.0 8.0 -12.0; 6.0 -12.0 18.0; 8.0 -16.0 24.0]
    zygote: value=[4.0, 9.0, 16.0], grad=[-4.0 8.0 -12.0; 6.0 -12.0 18.0; 8.0 -16.0 24.0]
